In [1]:
import os

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_file: Path
    local_data_file: Path


In [3]:
%pwd

'd:\\Projects\\RAG_Systems\\ProductDemand\\product-demand-forecasting-system\\research'

In [4]:
%pwd
os.chdir("..")

In [5]:
import sys
from pathlib import Path
from src.productdemand.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.productdemand.utils.common import read_yaml, create_directories
from src.productdemand.exception.custom_exception import CustomException

class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        try:
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)
            self.schema = read_yaml(schema_filepath)

             
            create_directories([Path(self.config.artifacts_root)])
        except Exception as e:
            raise CustomException(e, sys)

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        try:
            config = self.config.data_ingestion
            create_directories([Path(config.root_dir)])

            data_ingestion_config = DataIngestionConfig(
                root_dir=Path(config.root_dir),
                source_file=Path(config.source_file),
                local_data_file=Path(config.local_data_file)
            )

            return data_ingestion_config
        except Exception as e:
            raise CustomException(e, sys)


In [ ]:
import os
import sys
import shutil
from pathlib import Path
from src.productdemand.logger.custom_logger import get_logger
from src.productdemand.exception.custom_exception import CustomException
# from src.productdemand.entity.config_entity import DataIngestionConfig

logger = get_logger(__name__)

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def initiate_data_ingestion(self) -> None:
        """
        Copies the source local dataset to the artifacts data ingestion directory.
        """
        try:
            source_path = Path(self.config.source_file)
            local_path = Path(self.config.local_data_file)

            
            if not source_path.exists():
                raise FileNotFoundError(f"Source file not found at: {source_path.resolve()}")

          
            if not local_path.exists():
                logger.info("Copying local dataset to artifacts directory", 
                            source=str(source_path), 
                            destination=str(local_path))
                
                shutil.copy(src=source_path, dst=local_path)
                logger.info("Dataset ingested successfully", destination=str(local_path))
            else:
                logger.info("File already exists in ingestion directory, skipping copy", 
                            destination=str(local_path))

        except Exception as e:
            raise CustomException(e, sys)


In [7]:
import sys
# from src.productdemand.config.configuration import ConfigurationManager
# from src.productdemand.components.data_ingestion import DataIngestion
from src.productdemand.logger.custom_logger import get_logger
from src.productdemand.exception.custom_exception import CustomException

logger = get_logger(__name__)

STAGE_NAME = "Data Ingestion Stage"
try:
        logger.info(f">>>>>> Stage {STAGE_NAME} started <<<<<<")
        
        config = ConfigurationManager()
        data_ingestion_config = config.get_data_ingestion_config()
        data_ingestion = DataIngestion(config=data_ingestion_config)
        data_ingestion.initiate_data_ingestion()
        
        logger.info(f">>>>>> Stage {STAGE_NAME} completed successfully <<<<<<\n\nx==========x")
        
except Exception as e:
        logger.error(f"Stage {STAGE_NAME} failed")
        raise CustomException(e, sys)

{"timestamp": "2026-06-23T07:56:00.413506Z", "level": "info", "event": ">>>>>> Stage Data Ingestion Stage started <<<<<<"}
{"path": "config\\config.yaml", "timestamp": "2026-06-23T07:56:00.422994Z", "level": "info", "event": "YAML file loaded successfully"}
{"path": "params.yaml", "timestamp": "2026-06-23T07:56:00.426564Z", "level": "info", "event": "YAML file loaded successfully"}
{"path": "schema.yaml", "timestamp": "2026-06-23T07:56:00.429962Z", "level": "info", "event": "YAML file loaded successfully"}
{"path": "artifacts", "timestamp": "2026-06-23T07:56:00.432952Z", "level": "info", "event": "Created directory"}
{"path": "artifacts\\data_ingestion", "timestamp": "2026-06-23T07:56:00.435543Z", "level": "info", "event": "Created directory"}
{"source": "Retail_Dataset2.csv", "destination": "artifacts\\data_ingestion\\Retail_Dataset2.csv", "timestamp": "2026-06-23T07:56:00.438449Z", "level": "info", "event": "Copying local dataset to artifacts directory"}
{"destination": "artifacts\\d

In [8]:
import pandas as pd

df = pd.read_csv("Retail_Dataset2.csv")
print("Columns in CSV:")
print(list(df.columns))

Columns in CSV:
['Product_id', 'Product_Code', 'Warehouse', 'Product_Category', 'Date', 'Order_Demand', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Petrol_price']
